In [58]:
import warnings
warnings.filterwarnings('ignore')

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process

In [60]:
pd.set_option('display.max_columns', 500)

In [61]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} 

abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))
# https://gist.github.com/rogerallen/1583593

In [62]:
url = "https://www.nytimes.com/newsgraphics/polls/senate.csv"
polls = pd.read_csv(url)
polls.to_csv('../../2026_data/polls/senate.csv')
polls = polls[
    (polls['state'] != 'US') &
    (polls['stage'] == 'general')
]
polls.head()

,poll_id,pollster_id,pollster,sponsor_ids,sponsors,display_name,pollster_rating_id,pollster_rating_name,numeric_grade,pollscore,methodology,transparency_score,state,start_date,end_date,sponsor_candidate_id,sponsor_candidate,sponsor_candidate_party,question_id,sample_size,population,subpopulation,population_full,tracking,created_at,notes,url,url_article,url_topline,url_crosstab,source,internal,partisan,cycle,office_type,seat_name,seat_number,election_date,stage,party,pct,answer,candidate_name,candidate_id,race_id,ranked_choice_round,ranked_choice_reallocated,ranked_choice_final,nationwide_match,hypothetical
0,e05c43ca-5439-4bac-8148-b5a35785ebbd,312f97da-ebc5-446e-8856-cdded6ea1251,Big Data Poll,b519a86f-dbb8-46a2-8215-6abb6f6f389d,Public Polling Project,Big Data Poll,251.0,Big Data Poll,NaN,NaN,Text-to-Web/Nonprobability Panel/Live Phone,NaN,MI,9/22/26,9/24/26,NaN,NaN,NaN,6545327b-8ca1-4643-b788-618e9df011ff,725.0,rv,NaN,rv,NaN,9/25/26 19:16,NaN,https://www.bigdatapoll.com/blog/wolverine-sta...,NaN,NaN,NaN,NaN,NaN,NaN,2026,U.S. Senate,NaN,NaN,2026-11-03,general,REP,41.0,Rogers (Mike),Mike Rogers,7d9e51b3-0abf-43eb-8e48-5e03788745f6,8069ba7e-4ac6-449d-ae1e-f7fc5dee4a19,NaN,False,False,NaN,NaN
1,e05c43ca-5439-4bac-8148-b5a35785ebbd,312f97da-ebc5-446e-8856-cdded6ea1251,Big Data Poll,b519a86f-dbb8-46a2-8215-6abb6f6f389d,Public Polling Project,Big Data Poll,251.0,Big Data Poll,NaN,NaN,Text-to-Web/Nonprobability Panel/Live Phone,NaN,MI,9/22/26,9/24/26,NaN,NaN,NaN,6545327b-8ca1-4643-b788-618e9df011ff,725.0,rv,NaN,rv,NaN,9/25/26 19:16,NaN,https://www.bigdatapoll.com/blog/wolverine-sta...,NaN,NaN,NaN,NaN,NaN,NaN,2026,U.S. Senate,NaN,NaN,2026-11-03,general,DEM,45.4,El-Sayed,Abdul El-Sayed,2a1df6eb-59e4-4618-bf83-69c693f46fb2,8069ba7e-4ac6-449d-ae1e-f7fc5dee4a19,NaN,False,False,NaN,NaN
2,e05c43ca-5439-4bac-8148-b5a35785ebbd,312f97da-ebc5-446e-8856-cdded6ea1251,Big Data Poll,b519a86f-dbb8-46a2-8215-6abb6f6f389d,Public Polling Project,Big Data Poll,251.0,Big Data Poll,NaN,NaN,Text-to-Web/Nonprobability Panel/Live Phone,NaN,MI,9/22/26,9/24/26,NaN,NaN,NaN,bd20db7f-0151-4594-9a57-66c6f717e312,678.0,lv,NaN,lv,NaN,9/25/26 19:16,NaN,https://www.bigdatapoll.com/blog/wolverine-sta...,NaN,NaN,NaN,NaN,NaN,NaN,2026,U.S. Senate,NaN,NaN,2026-11-03,general,REP,42.1,Rogers (Mike),Mike Rogers,7d9e51b3-0abf-43eb-8e48-5e03788745f6,8069ba7e-4ac6-449d-ae1e-f7fc5dee4a19,NaN,False,False,NaN,NaN
3,e05c43ca-5439-4bac-8148-b5a35785ebbd,312f97da-ebc5-446e-8856-cdded6ea1251,Big Data Poll,b519a86f-dbb8-46a2-8215-6abb6f6f389d,Public Polling Project,Big Data Poll,251.0,Big Data Poll,NaN,NaN,Text-to-Web/Nonprobability Panel/Live Phone,NaN,MI,9/22/26,9/24/26,NaN,NaN,NaN,bd20db7f-0151-4594-9a57-66c6f717e312,678.0,lv,NaN,lv,NaN,9/25/26 19:16,NaN,https://www.bigdatapoll.com/blog/wolverine-sta...,NaN,NaN,NaN,NaN,NaN,NaN,2026,U.S. Senate,NaN,NaN,2026-11-03,general,DEM,46.7,El-Sayed,Abdul El-Sayed,2a1df6eb-59e4-4618-bf83-69c693f46fb2,8069ba7e-4ac6-449d-ae1e-f7fc5dee4a19,NaN,False,False,NaN,NaN
4,8f08ac98-5fe9-4052-a192-4157657541eb,7c25f6c3-cc63-4614-aec1-dfc392ef7872,Opinion Diagnostics,b8f5cd64-907b-45a3-ad37-256031be79f6,Common Cause North Carolina,Opinion Diagnostics,769.0,Opinion Diagnostics,NaN,NaN,Live Phone/Text-to-Web,NaN,NC,9/17/26,9/20/26,NaN,NaN,NaN,9cf6ab50-ec7a-4b46-8274-84effc1c2c0d,857.0,lv,NaN,lv,NaN,9/25/26 17:22,NaN,https://www.commoncause.org/north-carolina/pre...,NaN,NaN,NaN,NaN,NaN,NaN,2026,U.S. Senate,NaN,NaN,2026-11-03,general,NONE,2.8,Don't know,Don't know,d8390906-54c4-4050-9005-4934a65fa7b1,70395c5f-9aee-4a8f-a647-b656d0f894a3,NaN,False,False,NaN,NaN


In [63]:
polls_pivot = pd.pivot_table(data=polls, values='pct', index=['poll_id', 'question_id', 'state'], columns='candidate_name', aggfunc='first')
polls_pivot = polls_pivot.reset_index()
polls_pivot.head()

candidate_name,poll_id,question_id,state,Aaron Day,Abdul El-Sayed,Adam Hamilton,Alani Bankhead,Alexander Vindman,Allen Buckley,Amy McGrath,Andy Barr,Angie Craig,Angie Nixon,Ann Diener,Annie Andrews,Ashley Hinson,Ashley Moody,Barry Moore,Ben Ray Luján,Beto O’Rourke,Bill Hagerty,Bill Huizenga,Bill Redpath,Brad Raffensperger,Brian Bengs,Brian Kemp,Buddy Carter,Camencia Ford,Charles Booker,Charles D. Baker,Chris Pappas,Chris Sununu,Christine Lopez,Christopher Miklos,Cindy Burbank,Cindy Hyde-Smith,Colin Allred,Curtis Stinnett,Dan Innis,Dan J. Sullivan,Dan Kleban,Dan Osborn,Dan Sullivan,Daniel Cameron,Darline Graham,David Graham,David Roth,Derek Dooley,Don Brown,Don't know,Don't know/Someone else,Don't know/Would not vote,Douglas Marsh,Dustin Darden,Ed Markey,Edmond LaPlante,Edmond Laplante,Everett Wess,Frank Edelblut,Generic Candidate,Generic Democrat,Generic Republican,Gerald Heikes,Glenn Youngkin,Graham Platner,Gregory Levy,Gretchen Whitmer,Haley Stevens,Hallie Shoffner,Hans Truelson,Hector Mujica,Jack Reed,Jackie Norris,James Davis,James Risch,James Ryan,James Talarico,Jamie Davis,Janet Mills,Jasmine Crockett,Jeanne Logan Morrow,Jeanne Shaheen,Jeff Wadlin,Jennifer Jenkins,Jim Priest,Joaquin Castro,Joe Tache,John Cornyn,John Deaton,John Fetterman,John King,John Sununu,John Warren,Jon Husted,Jon Ossoff,Joni Ernst,Jordan Wood,Josh Turek,Joshua Cain,Julia Letlow,Julian Beaudion,Kasie Whitener,Kelly Loeffler,Ken Paxton,Kevin Hern,Kurt Alme,Kyle Austin,Lara Trump,Larry Marker,Lindsey Graham,Lydia Christensen,Mallory McMorrow,Marisa Simonetti,Marjorie Taylor Greene,Mark Lynch,Mark Sanford,Mark Warner,Marquita Bradshaw,Mary Peltola,Matt Giovonizzi,Matt Loesby,Michael Bahry,Michael Dublin,Michael Whatley,Michele Tafoya,Mike Collins,Mike Rogers,Mike Rounds,N'Kiyla Jasmine Thomas,Nancy Mace,Natalie Fleming,Nate Morris,Nathan Sage,Neil Gillespie,Neither,Nirav Shah,Noah Taylor,None/other,Other named candidates,Pamela Evette,Patrick Schmidt,Peggy Flanagan,Pete Buttigieg,Pete Ricketts,Ralph Norman,Raymond McKay,Rebecca Whiting,Reilly Neill,Rich McCormick,Roger Marshall,Ron Meinhardt,Roy Cooper,Russell Fry,Scott Brown,Scott Colom,Seth Bodnar,Seth Moulton,Sevier White,Shannon Bray,Shenna Bellows,Sherrod Brown,Sidney Hill,Someone else,Steve Daines,Susan Collins,Ted Brown,Thom Tillis,Thomas Laehn,Tim Harris,Tim Ryan,Timothy Long,Todd Achilles,Tom Cotton,Tom Jandron,Trey Gowdy,Troy Jackson,Ty Pinkins,Walter Kristy,Wesley Hunt,William J. Forbes,Would not vote,Zach Wahls
0,006eec7f-5ceb-4b68-8c21-658e35d6bb9a,8371ba4e-f777-48e0-a65b-904ffc8ff19c,TX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,007278d5-49cd-4aa2-9400-8c95c5fc50c5,071b6561-2fc5-4c00-98dd-4119da46c8d9,MI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [64]:
candinfo_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTZTum7NEaS6hvniyNJbDTWLDY0rATtgR5yXV2EUbtt6ubRPqv1sVH3PdJM3rgAzS_ak_XSgxLtaq3-/pub?output=csv'
candinfo = pd.read_csv(candinfo_url)

In [65]:
dem_uncont = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_uncont = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]
candinfo['state_po'] = candinfo['state'].astype(str).map(us_state_to_abbrev)
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po
0,Alabama,Everett Weiss,Barry Moore,False,False,AL
1,Alaska,Mary Peltola,Dan Sullivan,False,True,AK
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO
4,Delaware,Chris Coons,Michael Katz,True,False,DE


In [66]:
candinfo

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po
0,Alabama,Everett Weiss,Barry Moore,False,False,AL
1,Alaska,Mary Peltola,Dan Sullivan,False,True,AK
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO
4,Delaware,Chris Coons,Michael Katz,True,False,DE
5,Florida,Angie Nixon,Ashley Moody,False,True,FL
6,Georgia,Jon Ossoff,Mike Collins,True,False,GA
7,Idaho,Todd Achilles,Jim Risch,False,True,ID
8,Illinois,Juliana Stratton,Don Tracy,False,False,IL
9,Iowa,Josh Turek,Ashley Hinson,False,False,IA


In [67]:
dem_uncont.shape, rep_uncont.shape

((0, 5), (0, 5))

In [68]:
polled_cands = polls_pivot.columns.values[3:]
running_dems = np.unique(candinfo['dem_cand'].astype(str))
running_reps = np.unique(candinfo['rep_cand'].astype(str))
all_running_cands = np.concatenate([running_dems, running_reps])

In [69]:
all_running_cands

array(['Abdul El-Sayed', 'Adam Hamilton', 'Angie Nixon', 'Annie Andrews',
       'Ben Ray Lujan', 'Brian Bengs', 'Charles Booker', 'Chris Coons',
       'Chris Pappas', 'Cory Booker', 'Dan Osborn', 'Ed Markey',
       'Everett Weiss', 'Hallie Shoffner', 'Jack Reed', 'James Talarico',
       'James W. Byrd', 'Jamie Davis', 'Jeff Merkley',
       'John Hickenlooper', 'Jon Ossoff', 'Josh Turek',
       'Juliana Stratton', 'Mark Warner', 'Marquita Bradshaw',
       'Mary Peltola', "N'Kiyla Thomas", 'Peggy Flanagan',
       'Rachel Fetty Anderson', 'Roy Cooper', 'Scott Colom',
       'Seth Bodnar', 'Sherrod Brown', 'Todd Achilles', 'Troy Jackson',
       'Andy Barr', 'Ashley Hinson', 'Ashley Moody', 'Barry Moore',
       'Bert Mizusawa', 'Bill Hagerty', 'Cindy Hyde-Smith',
       'Dan Sullivan', 'Darline Graham', 'David Brock Smith', 'Don Tracy',
       'Harriet Hageman', 'Jim Risch', 'John Deaton', 'John Sununu',
       'Jon Husted', 'Julia Letlow', 'Justin Murphy', 'Ken Paxton',
       'K

In [70]:
hypo_cands = []
for c in polled_cands:
    fuzzymatch = process.extractOne(c, all_running_cands, scorer=fuzz.token_sort_ratio, score_cutoff=80)
    if fuzzymatch is None:
        hypo_cands.append(c)

In [71]:
hypo_cands

['Aaron Day',
 'Alani Bankhead',
 'Alexander Vindman',
 'Allen Buckley',
 'Amy McGrath',
 'Angie Craig',
 'Ann Diener',
 'Beto O’Rourke',
 'Bill Huizenga',
 'Bill Redpath',
 'Brad Raffensperger',
 'Brian Kemp',
 'Buddy Carter',
 'Camencia Ford',
 'Chris Sununu',
 'Christine Lopez',
 'Christopher Miklos',
 'Cindy Burbank',
 'Colin Allred',
 'Curtis Stinnett',
 'Dan Innis',
 'Dan Kleban',
 'Daniel Cameron',
 'David Graham',
 'David Roth',
 'Derek Dooley',
 'Don Brown',
 "Don't know",
 "Don't know/Someone else",
 "Don't know/Would not vote",
 'Douglas Marsh',
 'Dustin Darden',
 'Edmond LaPlante',
 'Edmond Laplante',
 'Frank Edelblut',
 'Generic Candidate',
 'Generic Democrat',
 'Generic Republican',
 'Gerald Heikes',
 'Glenn Youngkin',
 'Graham Platner',
 'Gregory Levy',
 'Gretchen Whitmer',
 'Haley Stevens',
 'Hans Truelson',
 'Hector Mujica',
 'Jackie Norris',
 'James Ryan',
 'Janet Mills',
 'Jasmine Crockett',
 'Jeanne Logan Morrow',
 'Jeanne Shaheen',
 'Jeff Wadlin',
 'Jennifer Jenkin

In [72]:
for h in hypo_cands:
    if h in ["N'Kiyla Jasmine Thomas",  "Don't know",
 "Don't know/Someone else",
 "Don't know/Would not vote", 'Someone else', 'Would not vote']:
        continue
    # print(h, polls_pivot["Christina Bohannan"].isna().all())
    polls_pivot = polls_pivot[polls_pivot[h].isna()]
    polls_pivot = polls_pivot.drop([h], axis=1)

for h in ['Dan J. Sullivan']: # names that weren't caught in hypo_cands but should've been
    polls_pivot = polls_pivot[polls_pivot[h].isna()]
    polls_pivot = polls_pivot.drop([h], axis=1)

In [73]:
polls_pivot.columns.values

array(['poll_id', 'question_id', 'state', 'Abdul El-Sayed',
       'Adam Hamilton', 'Andy Barr', 'Angie Nixon', 'Annie Andrews',
       'Ashley Hinson', 'Ashley Moody', 'Barry Moore', 'Ben Ray Luján',
       'Bill Hagerty', 'Brian Bengs', 'Charles Booker',
       'Charles D. Baker', 'Chris Pappas', 'Cindy Hyde-Smith',
       'Dan Osborn', 'Dan Sullivan', 'Darline Graham', "Don't know",
       "Don't know/Someone else", "Don't know/Would not vote",
       'Ed Markey', 'Everett Wess', 'Hallie Shoffner', 'Jack Reed',
       'James Davis', 'James Risch', 'James Talarico', 'Jamie Davis',
       'John Deaton', 'John Sununu', 'Jon Husted', 'Jon Ossoff',
       'Josh Turek', 'Julia Letlow', 'Ken Paxton', 'Kevin Hern',
       'Kurt Alme', 'Larry Marker', 'Mark Warner', 'Marquita Bradshaw',
       'Mary Peltola', 'Michael Whatley', 'Michele Tafoya',
       'Mike Collins', 'Mike Rogers', 'Mike Rounds',
       "N'Kiyla Jasmine Thomas", 'Peggy Flanagan', 'Pete Ricketts',
       'Raymond McKay', 'Ro

In [74]:
polls_pivot.shape

(250, 65)

In [75]:
rel_polls = polls[(polls['poll_id'].isin(np.unique(polls_pivot['poll_id']))) &
    (polls['question_id'].isin(np.unique(polls_pivot['question_id'])))]

In [76]:
rel_polls.shape

(856, 50)

In [77]:
np.unique(rel_polls['candidate_name'])

array(['Abdul El-Sayed', 'Adam Hamilton', 'Andy Barr', 'Angie Nixon',
       'Annie Andrews', 'Ashley Hinson', 'Ashley Moody', 'Barry Moore',
       'Ben Ray Luján', 'Brian Bengs', 'Charles Booker',
       'Charles D. Baker', 'Chris Pappas', 'Cindy Hyde-Smith',
       'Dan Osborn', 'Dan Sullivan', 'Darline Graham', "Don't know",
       "Don't know/Someone else", "Don't know/Would not vote",
       'Ed Markey', 'Everett Wess', 'Hallie Shoffner', 'Jack Reed',
       'James Davis', 'James Risch', 'James Talarico', 'Jamie Davis',
       'John Deaton', 'John Sununu', 'Jon Husted', 'Jon Ossoff',
       'Josh Turek', 'Julia Letlow', 'Ken Paxton', 'Kurt Alme',
       'Larry Marker', 'Mary Peltola', 'Michael Whatley',
       'Michele Tafoya', 'Mike Collins', 'Mike Rogers', 'Mike Rounds',
       'Peggy Flanagan', 'Pete Ricketts', 'Raymond McKay',
       'Roger Marshall', 'Roy Cooper', 'Scott Colom', 'Seth Bodnar',
       'Sherrod Brown', 'Someone else', 'Susan Collins', 'Todd Achilles',
       '

In [78]:
rel_polls.to_csv('transformed/relevent_senate_polls.csv')